# 🚀 Dashboard de Investimentos B3
### Célula 1 — Setup: Mock numba + Imports

In [1]:
import sys
import json
import warnings
warnings.filterwarnings('ignore')

# ── Mock numba (não instalado no Termux) ─────────────────────────────────────
# pandas_ta importa numba diretamente em _math.py.
# njit precisa ser um decorator transparente (retorna a própria função).
if 'numba' not in sys.modules:
    from unittest.mock import MagicMock
    _mock = MagicMock()
    _mock.njit = lambda f=None, **kw: (f if f else lambda fn: fn)
    _mock.prange = range
    sys.modules['numba']                  = _mock
    sys.modules['numba.core']             = MagicMock()
    sys.modules['numba.typed']            = MagicMock()
    sys.modules['numba.np']               = MagicMock()
    sys.modules['numba.np.numpy_support'] = MagicMock()
    print('✅ Mock numba aplicado')
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import plotly.graph_objs as go
import plotly.io as pio
from plotly.subplots import make_subplots
from datetime import datetime
from IPython.display import display, HTML

pio.renderers.default = 'notebook'  # Renderiza gráficos inline no Jupyter
print('✅ Imports concluídos')

✅ Mock numba aplicado
✅ Imports concluídos


In [167]:
# Módulos do projeto
from dados_financeiros import ColetorDados
from analise_fundamentalista import AnaliseFundamentalista
from analise_tecnica import AnaliseTecnica
from gestao_carteira import GestaoCarteira

In [165]:
# Módulos do projeto
import importlib

import dados_financeiros
import analise_fundamentalista
import analise_tecnica
import gestao_carteira


In [166]:
importlib.reload(dados_financeiros)
importlib.reload(analise_fundamentalista)
importlib.reload(analise_tecnica)
importlib.reload(gestao_carteira)

<module 'gestao_carteira' from 'c:\\Users\\vinic\\invest\\gestao_carteira.py'>

### Inicialização: Dados Fundamentalistas

In [3]:
import fundamentus as fd

print('🔄 Carregando dados fundamentalistas (pode demorar ~10s)...')
try:
    dados_fund = fd.get_resultado_raw()
    
    #Normaliza colunas percentuais para decimal
    colunas_pct = [
            "Dív.Líq/ Patrim."
    #    "Div.Yield", "Mrg Ebit", "Mrg. Líq.", "ROE", "ROIC", "Cresc. Rec.5a"
    ]
    for col in colunas_pct:
        if col in dados_fund.columns:
            dados_fund[col] = pd.to_numeric(
                dados_fund[col].astype(str)
                .str.replace("%", "", regex=False)
                .str.replace(",", ".", regex=False)
                .str.strip()
                .replace('-', float('nan')),
                errors="coerce"
            )

except Exception as e:
    print(f"🚨 Erro ao obter dados fundamentalistas: {e}")
    dados_fund = pd.DataFrame()

if dados_fund is not None and not dados_fund.empty:
    print(f'✅ {len(dados_fund)} empresas carregadas.')  # Preview para confirmar estrutura
else:
    print('⚠️ Dados fundamentalistas indisponíveis. Análises fundamentalistas desabilitadas.')

🔄 Carregando dados fundamentalistas (pode demorar ~10s)...
✅ 994 empresas carregadas.


In [4]:
display(dados_fund)

Multiples,Cotação,P/L,P/VP,PSR,Div.Yield,P/Ativo,P/Cap.Giro,P/EBIT,P/Ativ Circ.Liq,EV/EBIT,...,Mrg Bruta,Mrg Ebit,Mrg. Líq.,Liq. Corr.,ROIC,ROE,Liq.2meses,Patrim. Líq,Dív.Líq/ Patrim.,Cresc. Rec.5a
papel,,,,,,,,,,,,,,,,,,,,,
AALR3,3.24,-6.13,0.46,0.398,0.0000,0.163,112.89,5.72,-0.44,9.93,...,"26,83%",0.0696,-0.0570,1.01,0.0313,-0.0754,224769.0,1.066930e+09,0.34,0.0234
ABCB3,0.00,0.00,0.00,0.000,0.0000,0.000,0.00,0.00,0.00,0.00,...,"0,00%",0.0000,0.0000,0.00,0.0000,0.1408,0.0,7.147590e+09,0.00,0.0566
ABCB4,24.07,6.23,0.88,0.000,0.1021,0.000,0.00,0.00,0.00,0.00,...,"0,00%",0.0000,0.0000,0.00,0.0000,0.1408,19281400.0,7.147590e+09,0.00,0.0566
ABEV3,15.69,15.88,2.74,2.804,0.0547,1.732,211.56,12.08,-27.17,11.27,...,"51,48%",0.2321,0.1822,1.03,0.2004,0.1728,434535000.0,9.012580e+10,-0.18,0.0458
ABYA3,4.91,-214.80,1.76,2.055,0.0000,0.527,1.98,19.96,-2.75,33.67,...,"32,03%",0.1029,-0.0096,2.09,0.0278,-0.0082,0.0,2.920600e+08,1.21,0.1641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WLMM4,19.30,8.86,0.89,0.239,0.0905,0.446,1.62,5.39,5.07,7.22,...,"13,02%",0.0443,0.0269,1.89,0.1032,0.1000,28496.1,7.931650e+08,0.30,0.1657
WMBY3,25.39,-19.30,2.87,0.836,0.0000,0.182,1.20,8.62,-1.50,25.02,...,"26,91%",0.0970,-0.0705,1.44,0.0239,-0.1486,0.0,2.124390e+08,5.46,-0.1448
WSON33,67.00,8.07,0.98,1.067,0.0000,0.400,13.41,2.42,-0.89,5.08,...,"52,58%",0.4414,0.1358,1.26,0.1836,0.1217,0.0,2.148530e+09,1.08,0.0523


In [46]:
def _col(df: pd.DataFrame, nome: str):
    """Retorna o nome de coluna correto, testando maiúsculas e minúsculas."""

    # Mapeamento centralizado: nome semântico → coluna real do fundamentus
    COL = {
        "dy":         ["dy","Div.Yield"],
        "cotacao":    ["cotacao","Cotação"],
        "pl":         ["pl", "P/L"],
        "pvp":        ["pvp","P/VP"],
        "roe":        ["roe","ROE"],
        "roic":       ["roic","ROIC"],
        "mrgebit":    ["mrgebit","Mrg Ebit"],
        "mrgliq":     ["mrgliq","Mrg. Líq."],
        "patrliq":    ["patrliq","Patrim. Líq"],
        "divbpatr":   ["divbpatr","Dív.Líq/ Patrim."],
        "liq2m":      ["liq2m","Liq.2meses"],
        "evebit":     ["evebit","EV/EBIT"],
        "evebitda":   ["evebitda","EV/EBITDA"],
        "c5y":        ["c5y","Cresc. Rec.5a"],
        "l2m":        ["l2m","Liq.2meses"],
        "lpa":        ["lpa","lpa"],   # nem sempre disponível
        "vpa":        ["vpa","vpa"],   # nem sempre disponível
    }
    candidatos = [COL[nome][0], COL[nome][0].lower(), COL[nome][0].upper(), COL[nome][1], COL[nome][1].lower(), COL[nome][1].upper()]
    for c in candidatos:
        if c in df.columns:
            return c
    return None

def colunas_exibir(colunas):
    """Retorna um dicionário de colunas a exibir, mapeando nome real → alias."""
    colunas_exibir = {} 
    for alias, col_key in colunas:
        real = _col(dados_fund, col_key)
        if real:
            colunas_exibir[real] = alias
    return colunas_exibir

In [34]:
# Pega o nome correto das colunas, testando variações comuns (maiúsculas, minúsculas, sinônimos) 
dy_col          = _col(dados_fund, "dy")
patrliq_col     = _col(dados_fund, "patrliq")
roe_col         = _col(dados_fund, "roe")
l2m_col         = _col(dados_fund, "l2m")
mrgebit_col     = _col(dados_fund, "mrgebit")
divbpatr_col    = _col(dados_fund, "divbpatr")

### 🏆 Ranking Dividend Yield (Top 20)

### O que é Análise por Múltiplos de Mercado
A Análise por Múltiplos é uma metodologia de valuation relativo: em vez de calcular o valor intrínseco absoluto de uma empresa (como no DCF), compara-se o preço de uma ação com métricas financeiras padronizadas para identificar se ela está cara ou barata em relação aos pares do mesmo setor.

A lógica central é: empresas semelhantes, no mesmo setor e com características comparáveis, devem ser precificadas de forma semelhante pelo mercado. Desvios relevantes são potenciais oportunidades — ou armadilhas.

| Múltiplo | Fórmula                            | O que mede                                               | Referência                                        |
| -------- | ---------------------------------- | -------------------------------------------------------- | ------------------------------------------------- |
| DY       | Dividendo por Ação / Preço         | % do preço retornado em dividendos ao ano                | Comparável à renda fixa                           |
| ROE      | Lucro Líquido / Patrimônio Líquido | Eficiência no uso do capital do acionista                | > 15% é considerado bom                           |
| P/L      | Preço / Lucro por Ação             | Quanto o mercado paga por R$1 de lucro                   | P/L < 10 = barato; > 20 = caro (depende do setor) |
| P/VP     | Preço / Valor Patrimonial por Ação | Se a ação negocia acima ou abaixo do patrimônio contábil | P/VP < 1 = possível desconto                      |
| Mrg EBIT | EBIT / Receita Líquida             | Rentabilidade operacional antes de juros e IR            | Quanto maior, mais eficiente                      |
| Mrg Líq  | Lucro Líquido / Receita Líquida    | Rentabilidade final para o acionista                     | Varia muito por setor                             |


Uma empresa é considerada de qualidade mínima, ROE > 15% + Mrg EBIT > 10% + P/VP < 2,5

### Risco de "value trap"
Existe algumas armadilhas a evitar no longo prazo:
- **DY extraordinário:** Quando acontecem, podem representar dividendos não recorrentes — provavelmente dividendo especial único ou liquidação de reservas. Comprar pelo DY histórico nesses casos é uma armadilha clássica.
- **Múltiplo baixo por deterioração:** P/L baixo pode não significar desconto e sim refletir queda de lucros futuros já precificados pelos analistas, não uma oportunidade. Sempre cruzar com a tendência das margens.
- **Selic alta como concorrente:** com Selic Alta (14,5% em 2026), toda ação precisa entregar um prêmio de risco justificável. 

In [56]:
#Critérios de filtragem
min_dy = 0.09
min_roe = 0.05
min_patrliq = 1000_000_000
min_transacoes = 100_000
if dy_col is None:
    print(f"❌ Coluna DY não encontrada. Colunas disponíveis: {list(dados_fund.columns)}")
else:
    filtros = dados_fund[dy_col] > min_dy   # DY > min_dy 

if roe_col:
    filtros &= dados_fund[roe_col] > min_roe  # ROE > min_roe

if patrliq_col:
    filtros &= dados_fund[patrliq_col] > min_patrliq   # Patrimônio Líquido > min_patrliq

if l2m_col:
    filtros &= dados_fund[l2m_col] > min_transacoes  # Liquidez > min_transacoes

boas_pagadoras = dados_fund[filtros].copy()
print(f'✅ {len(boas_pagadoras)} empresas aprovadas no Ranking de Múltiplos')

✅ 39 empresas aprovadas no Ranking de Múltiplos


In [57]:
top_n = 20
ranking = boas_pagadoras.sort_values(dy_col, ascending=False).head(top_n)

# Monta DataFrame de exibição com as colunas disponíveis
colunas = colunas_exibir([
        ("Cotação",    "cotacao"),
        ("DY",         "dy"),
        ("ROE",        "roe"),
        ("P/L",        "pl"),
        ("P/VP",       "pvp"),
        ("Mrg EBIT",   "mrgebit"),
        ("Mrg Liq",    "mrgliq"),
        ("Patrim Liq", "patrliq"),
        ("Liq. 2 meses", "l2m"),
    ])
ranking_df = ranking[list(colunas.keys())].rename(columns=colunas)

# Formatos dinâmicos: só aplica formato se a coluna existir
fmt = {}
if "Cotação"   in ranking_df.columns: fmt["Cotação"]   = "R$ {:.2f}"
if "DY"        in ranking_df.columns: fmt["DY"]        = "{:.2%}"
if "ROE"       in ranking_df.columns: fmt["ROE"]       = "{:.2%}"
if "P/L"       in ranking_df.columns: fmt["P/L"]       = "{:.2f}"
if "P/VP"      in ranking_df.columns: fmt["P/VP"]      = "{:.2f}"
if "Mrg EBIT"  in ranking_df.columns: fmt["Mrg EBIT"]  = "{:.2%}"
if "Mrg Liq"   in ranking_df.columns: fmt["Mrg Liq"]   = "{:.2%}"
if "Patrim Liq" in ranking_df.columns:
    fmt["Patrim Liq"] = lambda x: (
        f"R$ {x/1_000_000_000:.2f} bi" if x >= 1_000_000_000
        else f"R$ {x/1_000_000:.2f} mi" if x >= 1_000_000
        else f"R$ {x:,.0f}"
    )
if "Liq. 2 meses" in ranking_df.columns:
    fmt["Liq. 2 meses"] = lambda x: (
        f"{x/1_000_000_000:.2f} bi" if x >= 1_000_000_000
        else f"{x/1_000_000:.2f} mi" if x >= 1_000_000
        else f"{x:,.0f}"
    )


display(
    ranking_df.style
    .format(fmt)
    .background_gradient(
        subset=['DY'] if 'DY' in ranking_df.columns else None,
        cmap='Greens'
    )
    .set_caption('Empresas aprovadas — Método Múltiplos')
)

Multiples,Cotação,DY,ROE,P/L,P/VP,Mrg EBIT,Mrg Liq,Patrim Liq,Liq. 2 meses
papel,,,,,,,,,
ALLD3,R$ 5.80,52.01%,22.27%,1.63,0.36,2.68%,6.26%,R$ 1.54 bi,3.21 mi
RIAA3,R$ 8.46,38.46%,28.33%,2.82,0.80,9.79%,14.20%,R$ 5.32 bi,17.88 mi
GRND3,R$ 3.99,36.99%,19.96%,5.68,1.13,9.59%,24.80%,R$ 3.17 bi,18.55 mi
VULC3,R$ 14.86,29.45%,45.47%,4.15,1.89,17.17%,31.34%,R$ 2.51 bi,25.48 mi
CMIN3,R$ 4.72,23.08%,31.49%,11.50,3.62,28.76%,12.51%,R$ 7.08 bi,41.26 mi
LAVV3,R$ 11.50,22.38%,29.03%,5.66,1.64,20.16%,25.00%,R$ 1.37 bi,11.17 mi
TRIS3,R$ 4.17,20.54%,12.92%,5.30,0.68,12.30%,13.68%,R$ 1.48 bi,2.58 mi
ALPA3,R$ 9.73,17.94%,17.90%,10.73,1.92,18.19%,13.15%,R$ 3.46 bi,"467,266"
DIRR3,R$ 12.98,17.08%,39.35%,8.06,3.17,24.77%,22.88%,R$ 2.13 bi,111.81 mi


### 🎯 Filtro Método Bazin

### O que é o Método Décio Bazin
Décio Bazin foi um jornalista financeiro e operador de bolsa brasileiro, autor do livro "Faça Fortuna com Ações, Antes que Seja Tarde" (1992). Seu método é uma abordagem majoritariamente quantitativa e matemática de seleção de ações com foco em renda passiva por dividendos.

### Os 3 pilares do Método Bazin:
**1. Dividend Yield (Cash Yield) ≥ 6% ao ano**
É o critério central. Bazin exigia que a ação pagasse pelo menos 6% a.a. em dividendos — calculado originalmente em dólar (Cash Yield), para filtrar distorções cambiais e inflacionárias. Se uma ação ficar 2 semestres consecutivos abaixo desse piso, deve ser vendida. O rebalanceamento ideal ocorre em abril e outubro (após os ciclos de dividendos).

  Regra de venda pelo preço:
  "Quando o DY cair para 4% (ação valoriza 50% desde a compra ao DY de 6%), o investidor pode vender e realizar o lucro." — Bazin, p.136
  Ou seja: compra a DY ≥ 6%, vende a DY ≈ 4%, embutindo automaticamente uma margem de segurança de ~50% de valorização.


   **⚠️ Crítica importante para o contexto atual — Selic a 14,5% a.a.:**
   Com a Selic em 14,5% ao ano (mai/2026), o critério original de 6% perde força como "barreira mínima de retorno". O Tesouro Selic hoje remunera mais do que o dobro do piso de Bazin com risco zero. Por isso, analistas e o próprio Bastter.com recomendam ajustar o piso do DY para pelo menos 60% da Selic (8-10% no cenário atual brasileiro), ou usar o rendimento do Tesouro + prêmio de risco como benchmark dinâmico — não um percentual fixo.


**2. Endividamento baixo (Dív/Patrim controlado)**
Bazin priorizava empresas com dívida líquida controlada. Empresas abaixo de 0.5 são empresas levemente alavancadas ou sem alavancagem.

**3. Ausência de notícias negativas relevantes**
O critério qualitativo: evitar empresas com problemas regulatórios, escândalos de gestão, mudanças abruptas de política de dividendos ou riscos sistêmicos identificáveis.

### DY x Bazin: Diferenças e Complementaridades
As duas metodologias se complementam: os múltiplos identificam o preço relativo e a qualidade operacional (margens, ROE), enquanto o Bazin filtra pela sustentabilidade dos dividendos e solidez financeira. Usados juntos, formam um filtro duplo que reduz o risco de "ciladas" — ações baratas por razões legítimas.

| Dimensão              | Análise por Múltiplos                         | Método Bazin                                   |
| --------------------- | --------------------------------------------- | ---------------------------------------------- |
| Objetivo              | Encontrar ações baratas vs. pares             | Encontrar pagadoras consistentes de dividendos |
| Indicador central     | P/L, P/VP, EV/EBITDA                          | Dividend Yield (Cash Yield)                    |
| Horizonte             | Pode ser curto/médio prazo                    | Exclusivamente longo prazo                     |
| Renda passiva         | Não é o foco                                  | Foco principal                                 |
| Risco de "value trap" | Alto (múltiplo baixo por problema estrutural) | Menor (DY sustentado exige empresa lucrativa)  |
| Abordagem             | Relativa (vs. setor)                          | Absoluta (piso de rendimento fixo)             |
| Critério qualitativo  | Baixo                                         | Presente (notícias negativas)                  |
| Complexidade          | Média-alta                                    | Baixa-média                                    |

### Encaixe nas Estratégia de Longo Prazo
Uma abordagem triangulada é o melhor caminho para o Value Investing de longo prazo, onde a paciência e a disciplina são os maiores diferenciadores.

A combinação ideal para o investidor de longo prazo na B3 segue esta lógica de 3 camadas sequenciais:

**CAMADA 1 — QUALIDADE (Múltiplos)**

ROE > 15% + Mrg EBIT > 10% + P/VP < 2,5
         
**CAMADA 2 — RENDA (Bazin adaptado)**

DY > Selic × 0,6 (ex: hoje ~8,7%) + Dív/Patrim < 0,5
         
**CAMADA 3 — MOMENTUM/TIMING (Análise Técnica)**

Entrada em suporte + tendência de alta confirmada


In [58]:
#Critérios de filtragem
selic = 0.145
min_mrgebit = 0.10
min_roe = 0.10
min_patrliq = 1_000_000
max_divbpatr = 0.5
min_transacoes = 500_000 # Exemplo: filtrar por liquidez mínima (1 milhão em transações nos últimos 2 meses)

if dy_col is None:
    print("❌ Coluna DY não encontrada.")

if dy_col is None:
    print(f"❌ Coluna DY não encontrada. Colunas disponíveis: {list(dados_fund.columns)}")
else:
    filtros = dados_fund[dy_col] > selic * 0.6

if mrgebit_col:
    filtros &= dados_fund[mrgebit_col] > min_mrgebit
if roe_col:
    filtros &= dados_fund[roe_col] > min_roe
if patrliq_col:
    filtros &= dados_fund[patrliq_col] > min_patrliq
if divbpatr_col:
    filtros &= dados_fund[divbpatr_col] < max_divbpatr
if l2m_col:
    filtros &= dados_fund[l2m_col] > min_transacoes 

empresas_bazin = dados_fund[filtros].copy()
empresas_bazin = empresas_bazin.sort_values(dy_col, ascending=False)
print(f'✅ {len(empresas_bazin)} empresas aprovadas no Método Bazin')

✅ 20 empresas aprovadas no Método Bazin


In [59]:
# Renomeia para exibição
colunas = colunas_exibir([
        ("Cotação",    "cotacao"),
        ("DY",         "dy"),
        ("ROE",        "roe"),
        ("Mrg EBIT",   "mrgebit"), 
        ("Dív/Patrim",   "divbpatr"), 
        ("P/L",        "pl"),
        ("P/VP",       "pvp"),
        ("Mrg Liq",    "mrgliq"),
        ("Patrim Liq", "patrliq"),
        ("Liq. 2 meses", "l2m"),
    ])
bazin_df = empresas_bazin[list(colunas.keys())].rename(columns=colunas)

if not bazin_df.empty:
    # Formatos dinâmicos: só aplica formato se a coluna existir
    fmt = {}
    if "Cotação"   in bazin_df.columns: fmt["Cotação"]   = "R$ {:.2f}"
    if "DY"        in bazin_df.columns: fmt["DY"]        = "{:.2%}"
    if "ROE"       in bazin_df.columns: fmt["ROE"]       = "{:.2%}"
    if "Mrg EBIT"  in bazin_df.columns: fmt["Mrg EBIT"]  = "{:.2%}"
    if "Dív/Patrim" in bazin_df.columns: fmt["Dív/Patrim"] = "{:.2f}"
    if "P/L"       in bazin_df.columns: fmt["P/L"]       = "{:.2f}"
    if "P/VP"      in bazin_df.columns: fmt["P/VP"]      = "{:.2f}"
    if "Mrg Liq"   in ranking_df.columns: fmt["Mrg Liq"]   = "{:.2%}"
    if "Patrim Liq" in ranking_df.columns:
        fmt["Patrim Liq"] = lambda x: (
            f"R$ {x/1_000_000_000:.2f} bi" if x >= 1_000_000_000
            else f"R$ {x/1_000_000:.2f} mi" if x >= 1_000_000
            else f"R$ {x:,.0f}"
        )
    if "Liq. 2 meses" in ranking_df.columns:
        fmt["Liq. 2 meses"] = lambda x: (
            f"{x/1_000_000_000:.2f} bi" if x >= 1_000_000_000
            else f"{x/1_000_000:.2f} mi" if x >= 1_000_000
            else f"{x:,.0f}"
        )

    display(
        bazin_df.style
        .format(fmt)
        .background_gradient(
            subset=['DY'] if 'DY' in bazin_df.columns else None,
            cmap='Greens'
        )
        .set_caption('Empresas aprovadas — Método Décio Bazin')
    )
else:
    print('⚠️ Nenhuma empresa passou nos critérios do Método Bazin.')

Multiples,Cotação,DY,ROE,Mrg EBIT,Dív/Patrim,P/L,P/VP,Mrg Liq,Patrim Liq,Liq. 2 meses
papel,,,,,,,,,,
VULC3,R$ 14.86,29.45%,45.47%,17.17%,0.27,4.15,1.89,31.34%,R$ 2.51 bi,25.48 mi
CMIN3,R$ 4.72,23.08%,31.49%,28.76%,0.04,11.50,3.62,12.51%,R$ 7.08 bi,41.26 mi
LAVV3,R$ 11.50,22.38%,29.03%,20.16%,0.35,5.66,1.64,25.00%,R$ 1.37 bi,11.17 mi
TRIS3,R$ 4.17,20.54%,12.92%,12.30%,0.31,5.30,0.68,13.68%,R$ 1.48 bi,2.58 mi
DIRR3,R$ 12.98,17.08%,39.35%,24.77%,0.35,8.06,3.17,22.88%,R$ 2.13 bi,111.81 mi
MELK3,R$ 3.21,16.35%,11.47%,12.63%,0.43,5.35,0.61,13.60%,R$ 1.08 bi,2.49 mi
MDNE3,R$ 27.40,15.92%,23.93%,22.22%,0.04,5.64,1.35,19.85%,R$ 2.12 bi,43.32 mi
ALPA4,R$ 11.29,15.90%,17.90%,18.19%,0.14,12.45,2.23,13.15%,R$ 3.46 bi,30.78 mi
POMO3,R$ 5.95,15.87%,30.72%,15.77%,0.38,5.96,1.83,13.91%,R$ 4.06 bi,5.30 mi


### Célula 3 — Carteira do Usuário

In [8]:
from gestao_carteira import GestaoCarteira

carteira_data = {
    'COGN3': {'qtd': 470,  'preco_compra': 27.90},
    'EGIE3': {'qtd': 200,  'preco_compra': 40.30},
    'BBAS3': {'qtd': 300,  'preco_compra': 22.90},
    'VALE3': {'qtd': 100,  'preco_compra': 50.10},
    'HBOR3': {'qtd': 1500, 'preco_compra':  2.96},
    'IRBR3': {'qtd': 1213, 'preco_compra':  3.17},
    'SIMH3': {'qtd': 600,  'preco_compra':  4.50},
    'AGRO3': {'qtd': 100,  'preco_compra': 20.00},
    'VAMO3': {'qtd': 400,  'preco_compra':  4.65},
    'GFSA3': {'qtd': 600,  'preco_compra':  2.50},
    'EZTC3': {'qtd': 300,  'preco_compra':  4.60},
    'VIVT3': {'qtd': 800,  'preco_compra':  1.50},
    'AMOB3': {'qtd': 450,  'preco_compra':  1.43},
}

carteira_usuario = GestaoCarteira()
for ticker, d in carteira_data.items():
    carteira_usuario.adicionar_ativo(ticker, d['qtd'], d['preco_compra'])

print(f'✅ Carteira carregada com {len(carteira_data)} ativos.')
print('Tickers:', ', '.join(carteira_data.keys()))

✅ Carteira carregada com 13 ativos.
Tickers: COGN3, EGIE3, BBAS3, VALE3, HBOR3, IRBR3, SIMH3, AGRO3, VAMO3, GFSA3, EZTC3, VIVT3, AMOB3


In [16]:
import yfinance as yf
import requests
import requests_cache
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

# Sessão com retry automático e User-Agent de navegador
session = requests.Session()
session.headers.update({
            "User-Agent": (
                "Mozilla/5.0 (Linux; Android 12; Pixel 6) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0 Mobile Safari/537.36"
            )
        })

retry = Retry(
            total=3,
            backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
        )
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

data = yf.download(
                    'COGN3.SA',
                    period="2d",
                    auto_adjust=True,
                    progress=False,
                    group_by="ticker",
                    session=session,
                  )

                   

2026-05-15 07:10:50,973 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'COGN3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 07:10:54,633 [multi.download] ERROR: 
1 Failed download:
2026-05-15 07:10:54,636 [multi.download] ERROR: ['COGN3.SA']: RetryError(MaxRetryError("HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))"))


In [ ]:
print(data)

### Célula 4 — Resumo da Carteira (Dashboard Principal)

In [11]:
import dados_financeiros

print('🔄 Buscando preços atuais...')
precos_atuais = coletor.obter_precos_em_lote(carteira_data.keys())
performance   = carteira_usuario.calcular_performance(precos_atuais)

total_investido = performance.get('TOTAL', {}).get('valor_investido', 0)
valor_atual     = performance.get('TOTAL', {}).get('valor_atual', 0)
lucro           = valor_atual - total_investido
rentabilidade   = (lucro / total_investido * 100) if total_investido else 0

dy_carteira = 0
if analise_fund is not None:
    dy_carteira = carteira_usuario.calcular_dividend_yield_carteira(dados_fund)

2026-05-15 06:45:29,728 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'VIVT3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:45:33,338 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'VAMO3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:45:37,003 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'AMOB3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:45:40,559 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'SIMH3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/get

⚠️ Download em lote vazio na tentativa 2.
⏳ Lote — tentativa 3/3, aguardando 5.8s...


2026-05-15 06:48:35,703 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'VALE3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:48:39,346 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'EGIE3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:48:43,028 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'EZTC3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/getcrumb (Caused by ResponseError('too many 429 error responses'))
2026-05-15 06:48:46,680 [base._fetch_ticker_tz] ERROR: Failed to get ticker 'VIVT3.SA' reason: HTTPSConnectionPool(host='query1.finance.yahoo.com', port=443): Max retries exceeded with url: /v1/test/get

⚠️ Download em lote vazio na tentativa 3.
❌ Falha após 3 tentativas. Retornando zeros.


KeyError: "None of [Index(['Div.Yield'], dtype='str', name='Multiples')] are in the [columns]"

In [10]:
cor_lucro = '#42b72a' if lucro >= 0 else '#fa3e3e'
sinal     = '+' if lucro >= 0 else ''

display(HTML(f'''
<div style='font-family:sans-serif; display:flex; gap:16px; flex-wrap:wrap;'>
  <div style='background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1); min-width:200px;'>
    <h3 style='color:#1877f2; margin:0 0 12px'>💰 Valor Investido</h3>
    <p style='font-size:24px; font-weight:bold; margin:0'>R$ {total_investido:,.2f}</p>
  </div>
  <div style='background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1); min-width:200px;'>
    <h3 style='color:#1877f2; margin:0 0 12px'>📈 Valor Atual</h3>
    <p style='font-size:24px; font-weight:bold; margin:0'>R$ {valor_atual:,.2f}</p>
  </div>
  <div style='background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1); min-width:200px;'>
    <h3 style='color:#1877f2; margin:0 0 12px'>🎯 Resultado</h3>
    <p style='font-size:24px; font-weight:bold; margin:0; color:{cor_lucro}'>{sinal}R$ {lucro:,.2f} ({sinal}{rentabilidade:.2f}%)</p>
  </div>
  <div style='background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1); min-width:200px;'>
    <h3 style='color:#1877f2; margin:0 0 12px'>💸 DY Médio Carteira</h3>
    <p style='font-size:24px; font-weight:bold; margin:0'>{dy_carteira:.2f}%</p>
  </div>
</div>
'''))

### Célula 7 — 📈 Análise Técnica
> Altere `TICKER` na linha abaixo e re-execute a célula.

In [17]:
# ── Fonte alternativa: brapi.dev (fallback quando Yahoo está bloqueado) ──────
import requests as _requests

def obter_historico_brapi(ticker: str, periodo: str = "1y") -> pd.DataFrame:
    """
    Busca histórico OHLCV via brapi.dev.
    Não precisa de API key. Cobre todos os tickers da B3.
    Períodos aceitos: 1d 5d 1mo 3mo 6mo 1y 2y 5y 10y ytd max
    """
    periodo_map = {
        "3mo": "3mo", "6mo": "6mo",
        "1y":  "1y",  "2y":  "2y",
    }
    range_param = periodo_map.get(periodo, "1y")
    url = (
        f"https://brapi.dev/api/quote/{ticker}"
        f"?range={range_param}&interval=1d&fundamental=false"
    )

    try:
        resp = _requests.get(url, timeout=20)
        resp.raise_for_status()
        data = resp.json()

        results = data.get("results", [])
        if not results:
            print(f"⚠️ brapi: sem resultados para {ticker}")
            return pd.DataFrame()

        hist = results[0].get("historicalDataPrice", [])
        if not hist:
            print(f"⚠️ brapi: sem histórico para {ticker}")
            return pd.DataFrame()

        df = pd.DataFrame(hist)

        # timestamp Unix → datetime
        df["date"] = pd.to_datetime(df["date"], unit="s", utc=True).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)
        df = df.set_index("date")
        df.index.name = "Date"

        df = df.rename(columns={
            "open":   "Open",
            "high":   "High",
            "low":    "Low",
            "close":  "Close",
            "volume": "Volume",
        })

        colunas = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in df.columns]
        df = df[colunas].dropna(subset=["Close"])
        df = df.sort_index()

        print(f"✅ brapi: {len(df)} candles carregados para {ticker}")
        return df

    except Exception as e:
        print(f"🚨 Erro brapi ({ticker}): {e}")
        return pd.DataFrame()


print("✅ Fonte alternativa brapi.dev pronta.")

✅ Fonte alternativa brapi.dev pronta.


In [18]:
# ── Configuração ─────────────────────────────
TICKER  = 'GRND3'   # <- altere aqui
PERIODO = '1y'       # 3mo | 6mo | 1y | 2y
# ─────────────────────────────────────────────

print(f'🔄 Buscando histórico de {TICKER}...')
#dados_hist = coletor.obter_historico_preco(TICKER, PERIODO)
dados_hist = obter_historico_brapi(TICKER, PERIODO)  
if dados_hist.empty:
    print(f'❌ Nenhum dado encontrado para {TICKER}. Verifique o código do ativo.')
else:
    analise_tec              = AnaliseTecnica(dados_hist)
    dados_ind                = analise_tec.calcular_indicadores()
    sinais                   = analise_tec.identificar_sinais()
    suporte, resistencia     = analise_tec.calcular_suporte_resistencia()

    # ── Gráfico principal (candlestick + médias) ──────────────────────────────
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        row_heights=[0.6, 0.2, 0.2],
        subplot_titles=(f'{TICKER} — Preço', 'RSI (14)', 'MACD')
    )

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=dados_ind.index,
        open=dados_ind['Open'], high=dados_ind['High'],
        low=dados_ind['Low'],   close=dados_ind['Close'],
        name='Preço', showlegend=False
    ), row=1, col=1)

    # Médias móveis
    for col, cor, nome in [('SMA_20','blue','SMA 20'), ('SMA_50','orange','SMA 50'), ('BB_Upper','gray','BB +'), ('BB_Lower','gray','BB -')]:
        if col in dados_ind.columns:
            fig.add_trace(go.Scatter(
                x=dados_ind.index, y=dados_ind[col],
                name=nome, line=dict(color=cor, width=1, dash='dot' if 'BB' in col else 'solid')
            ), row=1, col=1)

    # Suporte / Resistência
    if suporte:    fig.add_hline(y=suporte,    line_dash='dash', line_color='green', row=1, col=1)
    if resistencia: fig.add_hline(y=resistencia, line_dash='dash', line_color='red',   row=1, col=1)

    # RSI
    if 'RSI' in dados_ind.columns:
        fig.add_trace(go.Scatter(x=dados_ind.index, y=dados_ind['RSI'], name='RSI', line=dict(color='purple', width=1)), row=2, col=1)
        fig.add_hline(y=70, line_dash='dash', line_color='red',   row=2, col=1)
        fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)

    # MACD
    if 'MACD' in dados_ind.columns:
        fig.add_trace(go.Scatter(x=dados_ind.index, y=dados_ind['MACD'],        name='MACD',   line=dict(color='blue', width=1)), row=3, col=1)
        fig.add_trace(go.Scatter(x=dados_ind.index, y=dados_ind['MACD_Signal'], name='Signal', line=dict(color='orange', width=1)), row=3, col=1)
        fig.add_bar(x=dados_ind.index, y=dados_ind['MACD_Histogram'], name='Histograma', row=3, col=1)

    fig.update_layout(
        height=700, title=f'Análise Técnica — {TICKER}',
        xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', yanchor='bottom', y=1.02)
    )
    fig.show()

    # ── Sinais e níveis ────────────────────────────────────────────────────────
    sinais_txt = '<br>'.join([
        f"<span style='color:{'#42b72a' if 'COMPRA' in s else '#fa3e3e'}'>● {s}</span>"
        for s in sinais
    ]) or '<span style="color:#999">Nenhum sinal claro identificado.</span>'

    display(HTML(f'''
    <div style='font-family:sans-serif; background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1); margin-top:12px;'>
      <h3 style='color:#1877f2'>🎯 Sinais Identificados</h3>
      <p>{sinais_txt}</p>
      <h3 style='color:#1877f2'>📊 Níveis Chave</h3>
      <p><strong>Suporte:</strong> R$ {suporte:.2f if suporte else 'N/A'}</p>
      <p><strong>Resistência:</strong> R$ {resistencia:.2f if resistencia else 'N/A'}</p>
    </div>
    '''))

🔄 Buscando histórico de GRND3...
🚨 Erro brapi (GRND3): 401 Client Error: Unauthorized for url: https://brapi.dev/api/quote/GRND3?range=1y&interval=1d&fundamental=false
❌ Nenhum dado encontrado para GRND3. Verifique o código do ativo.


### Célula 8 — ⏰ Monitoramento da Carteira em Tempo Real

In [ ]:
print('🔄 Buscando dados em tempo real...')
dados_tr = coletor.monitorar_carteira_tempo_real(list(carteira_data.keys()))

rows = ''
for ticker, d in dados_tr.items():
    cor  = '#42b72a' if d['variacao_dia'] >= 0 else '#fa3e3e'
    sinal = '+' if d['variacao_dia'] >= 0 else ''
    rows += f"""
    <tr>
      <td><strong>{ticker}</strong></td>
      <td>R$ {d['preco']:.2f}</td>
      <td style='color:{cor}; font-weight:bold'>{sinal}{d['variacao_dia']:.2f}%</td>
      <td>{d['volume']:,.0f}</td>
    </tr>"""

display(HTML(f'''
<div style='font-family:sans-serif; background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1);'>
  <h3 style='color:#1877f2'>⏰ Preços em Tempo Real — {datetime.now().strftime('%d/%m/%Y %H:%M')}</h3>
  <table style='width:100%; border-collapse:collapse;'>
    <thead>
      <tr style='background:#f5f7fa;'>
        <th style='padding:10px; text-align:left;'>Ticker</th>
        <th style='padding:10px; text-align:left;'>Preço</th>
        <th style='padding:10px; text-align:left;'>Var. Dia</th>
        <th style='padding:10px; text-align:left;'>Volume</th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
</div>
'''))

### Célula 9 — 💼 Performance Detalhada da Carteira

In [ ]:
# Reutiliza precos_atuais da Célula 4 (re-execute a Célula 4 para atualizar)
perf_rows = ''
for ticker, d in performance.items():
    if ticker == 'TOTAL':
        continue
    cor   = '#42b72a' if d['lucro_prejuizo'] >= 0 else '#fa3e3e'
    sinal = '+' if d['lucro_prejuizo'] >= 0 else ''
    perf_rows += f"""
    <tr>
      <td><strong>{ticker}</strong></td>
      <td>{d['quantidade']}</td>
      <td>R$ {d['preco_compra']:.2f}</td>
      <td>R$ {d['preco_atual']:.2f}</td>
      <td>R$ {d['valor_investido']:,.2f}</td>
      <td>R$ {d['valor_atual']:,.2f}</td>
      <td style='color:{cor}; font-weight:bold'>{sinal}R$ {d['lucro_prejuizo']:,.2f}<br><small>{sinal}{d['rentabilidade']:.2f}%</small></td>
    </tr>"""

total     = performance.get('TOTAL', {})
cor_total = '#42b72a' if total.get('lucro_prejuizo', 0) >= 0 else '#fa3e3e'
sinal_tot = '+' if total.get('lucro_prejuizo', 0) >= 0 else ''

display(HTML(f'''
<div style='font-family:sans-serif; background:#fff; padding:20px; border-radius:8px; box-shadow:0 2px 4px rgba(0,0,0,.1);'>
  <h3 style='color:#1877f2'>💼 Performance da Carteira</h3>
  <table style='width:100%; border-collapse:collapse;'>
    <thead>
      <tr style='background:#f5f7fa; font-weight:600;'>
        <th style='padding:10px; text-align:left;'>Ticker</th>
        <th style='padding:10px; text-align:left;'>Qtd</th>
        <th style='padding:10px; text-align:left;'>Preço Médio</th>
        <th style='padding:10px; text-align:left;'>Preço Atual</th>
        <th style='padding:10px; text-align:left;'>Investido</th>
        <th style='padding:10px; text-align:left;'>Atual</th>
        <th style='padding:10px; text-align:left;'>Resultado</th>
      </tr>
    </thead>
    <tbody>
      {perf_rows}
      <tr style='background:#f5f7fa; font-weight:bold;'>
        <td colspan='4'>TOTAL</td>
        <td>R$ {total.get('valor_investido',0):,.2f}</td>
        <td>R$ {total.get('valor_atual',0):,.2f}</td>
        <td style='color:{cor_total}'>{sinal_tot}R$ {total.get('lucro_prejuizo',0):,.2f} ({sinal_tot}{total.get('rentabilidade',0):.2f}%)</td>
      </tr>
    </tbody>
  </table>
</div>
'''))